# STREAMSENSE — quantize_ptq.ipynb

Post-Training Quantization: FP32 ONNX -> INT8 ONNX

Place at: `C:\STREAMSENSE\quantize_ptq.ipynb`

Kernel: `streamsense-env-win`

Requirements: `pip install onnx onnxruntime`

Before running: confirm `onnx_models/streamsense_model_fp32.onnx` exists.

In [1]:
# Cell 1 - Imports and paths
import sys
import json
import shutil
import random
import numpy as np
import torch
import onnxruntime as ort
from pathlib import Path
from onnxruntime.quantization import (
    quantize_static,
    CalibrationDataReader,
    QuantType,
    QuantFormat,
)

ROOT         = Path(r'C:\STREAMSENSE')
sys.path.insert(0, str(ROOT / 'training'))

from mel_pipeline import preprocess

FP32_PATH    = ROOT / 'onnx_models'    / 'streamsense_model_fp32.onnx'
INT8_PATH    = ROOT / 'onnx_models'    / 'streamsense_model_int8.onnx'
CALIB_DIR    = ROOT / 'temp_calibration'
TRAIN_SPLIT  = ROOT / 'data' / 'splits' / 'train_files.txt'
GV_RAW_DIR   = ROOT / 'golden_vectors' / 'raw'
GV_NORM_DIR  = ROOT / 'golden_vectors' / 'normalized'
MANIFEST     = ROOT / 'golden_vectors' / 'manifest.json'

N_CALIB      = 1000   # number of calibration samples
RANDOM_SEED  = 42

print('=== Path Check ===')
for p, name in [
    (FP32_PATH,   'streamsense_model_fp32.onnx'),
    (TRAIN_SPLIT, 'train_files.txt'),
    (MANIFEST,    'manifest.json'),
]:
    print(f'  [{"OK" if p.exists() else "MISSING"}] {name}')

print(f'\nINT8 output -> {INT8_PATH}')

=== Path Check ===
  [OK] streamsense_model_fp32.onnx
  [OK] train_files.txt
  [OK] manifest.json

INT8 output -> C:\STREAMSENSE\onnx_models\streamsense_model_int8.onnx


In [2]:
# Cell 2 - Sample 1000 calibration files from train split
import torchaudio

print(f'Reading train split...')
with open(TRAIN_SPLIT) as f:
    lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]

print(f'  Total train samples : {len(lines)}')

random.seed(RANDOM_SEED)
selected = random.sample(lines, min(N_CALIB, len(lines)))
print(f'  Calibration samples : {len(selected)}')

# Build calibration tensors
CALIB_DIR.mkdir(exist_ok=True)
print(f'\nProcessing calibration samples through mel_pipeline...')

calib_tensors = []
skipped = 0

for i, line in enumerate(selected):
    parts = line.split('|')
    if len(parts) != 3:
        skipped += 1
        continue

    wav_path = Path(parts[0].strip())
    if not wav_path.exists():
        skipped += 1
        continue

    try:
        waveform, sr = torchaudio.load(str(wav_path))
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        raw    = waveform.squeeze(0).numpy()
        tensor = preprocess(raw)                    # [1, 1, 64, 97]
        x      = tensor.squeeze(0).unsqueeze(0)     # [1, 1, 64, 97]
        calib_tensors.append(x.numpy())
    except Exception:
        skipped += 1
        continue

    if (i + 1) % 200 == 0:
        print(f'  Processed {i+1}/{len(selected)}...')

print(f'\nCalibration tensors : {len(calib_tensors)}')
print(f'Skipped             : {skipped}')
print(f'Tensor shape        : {calib_tensors[0].shape}')
print(f'Tensor dtype        : {calib_tensors[0].dtype}')

Reading train split...
  Total train samples : 26984
  Calibration samples : 1000

Processing calibration samples through mel_pipeline...
  Processed 200/1000...
  Processed 400/1000...
  Processed 600/1000...
  Processed 800/1000...
  Processed 1000/1000...

Calibration tensors : 1000
Skipped             : 0
Tensor shape        : (1, 1, 64, 97)
Tensor dtype        : float32


In [3]:
# Cell 3 - Define calibration data reader
# ONNX Runtime quantizer needs a CalibrationDataReader object
# It iterates through calibration tensors one by one

class StreamSenseCalibReader(CalibrationDataReader):
    def __init__(self, tensors):
        self.tensors = tensors
        self.index   = 0

    def get_next(self):
        if self.index >= len(self.tensors):
            return None
        data = {'input': self.tensors[self.index]}
        self.index += 1
        return data

    def rewind(self):
        self.index = 0

calib_reader = StreamSenseCalibReader(calib_tensors)
print(f'CalibrationDataReader ready.')
print(f'  Samples : {len(calib_tensors)}')
print(f'  Input   : input -> {calib_tensors[0].shape} float32')

CalibrationDataReader ready.
  Samples : 1000
  Input   : input -> (1, 1, 64, 97) float32


In [4]:
# Cell 4 - Run static INT8 quantization
print('Running PTQ static quantization...')
print('This may take 2-5 minutes.')
print()

quantize_static(
    model_input          = str(FP32_PATH),
    model_output         = str(INT8_PATH),
    calibration_data_reader = calib_reader,
    quant_format         = QuantFormat.QDQ,     # Quantize-DeQuantize format
    per_channel          = False,               # per-tensor quantization
    weight_type          = QuantType.QInt8,     # INT8 weights
    activation_type      = QuantType.QInt8,     # INT8 activations
)

fp32_size = FP32_PATH.stat().st_size / 1e6
int8_size = INT8_PATH.stat().st_size / 1e6
reduction = (1 - int8_size / fp32_size) * 100

print(f'Quantization complete.')
print(f'  FP32 size : {fp32_size:.2f} MB')
print(f'  INT8 size : {int8_size:.2f} MB')
print(f'  Reduction : {reduction:.1f}%')

Running PTQ static quantization...
This may take 2-5 minutes.



Quantization complete.
  FP32 size : 1.19 MB
  INT8 size : 0.31 MB
  Reduction : 73.6%


In [5]:
# Cell 5 - Load manifest and golden vectors
with open(MANIFEST) as f:
    manifest = json.load(f)

tolerance = float(manifest['tolerance_max_abs_error'])
vectors   = manifest['vectors']

def load_bin(path, shape):
    arr = np.fromfile(str(path), dtype='<f4')
    return arr.reshape(shape)

gv_data = []
for i in range(10):
    v    = vectors[str(i)]
    raw  = load_bin(GV_RAW_DIR  / v['raw_bin'],  tuple(v['raw_shape']))
    norm = load_bin(GV_NORM_DIR / v['norm_bin'], tuple(v['mel_shape']))
    gv_data.append({'label': v['label'], 'raw': raw, 'norm': norm})

print(f'Golden vectors loaded: {len(gv_data)}')
print(f'Tolerance            : {tolerance}')

Golden vectors loaded: 10
Tolerance            : 0.0001


In [6]:
# Cell 6 - Validate INT8 model against golden vectors
# Compare FP32 predictions vs INT8 predictions
# top-1 must match for all 10 vectors

fp32_session = ort.InferenceSession(
    str(FP32_PATH), providers=['CPUExecutionProvider']
)
int8_session = ort.InferenceSession(
    str(INT8_PATH), providers=['CPUExecutionProvider']
)

print('=' * 60)
print('INT8 Validation - FP32 vs INT8 on Golden Vectors')
print('=' * 60)
print()

passed = 0
failed = 0

for i, gv in enumerate(gv_data):
    tensor = preprocess(gv['raw'])
    x      = tensor.squeeze(0).unsqueeze(0).numpy()    # [1, 1, 64, 97]

    fp32_logits = fp32_session.run(['logits'], {'input': x})[0]
    int8_logits = int8_session.run(['logits'], {'input': x})[0]

    fp32_pred   = int(np.argmax(fp32_logits))
    int8_pred   = int(np.argmax(int8_logits))
    top1_match  = fp32_pred == int8_pred
    logit_diff  = float(np.max(np.abs(fp32_logits - int8_logits)))

    if top1_match:
        passed += 1
    else:
        failed += 1

    status = 'PASS' if top1_match else 'FAIL'
    print(
        f'  [{status}] GV_{i:02d}_{gv["label"]:<6}  '
        f'FP32={fp32_pred}  INT8={int8_pred}  '
        f'top1_match={top1_match}  '
        f'logit_diff={logit_diff:.4f}'
    )

print()
print('=' * 60)
print(f'Passed : {passed}/10')
print(f'Failed : {failed}/10')
if failed == 0:
    print('[PASS] INT8 model validated. All top-1 predictions match FP32.')
else:
    print('[FAIL] Some top-1 predictions differ. Check logit_diff values.')
print('=' * 60)

INT8 Validation - FP32 vs INT8 on Golden Vectors

  [PASS] GV_00_yes     FP32=0  INT8=0  top1_match=True  logit_diff=0.2975
  [PASS] GV_01_no      FP32=1  INT8=1  top1_match=True  logit_diff=0.3824
  [PASS] GV_02_up      FP32=2  INT8=2  top1_match=True  logit_diff=0.1134
  [PASS] GV_03_down    FP32=3  INT8=3  top1_match=True  logit_diff=0.2378
  [PASS] GV_04_left    FP32=4  INT8=4  top1_match=True  logit_diff=0.4012
  [PASS] GV_05_right   FP32=5  INT8=5  top1_match=True  logit_diff=0.1849
  [PASS] GV_06_on      FP32=6  INT8=6  top1_match=True  logit_diff=0.2085
  [PASS] GV_07_off     FP32=7  INT8=7  top1_match=True  logit_diff=0.1167
  [PASS] GV_08_stop    FP32=8  INT8=8  top1_match=True  logit_diff=0.1990
  [PASS] GV_09_go      FP32=9  INT8=9  top1_match=True  logit_diff=0.1372

Passed : 10/10
Failed : 0/10
[PASS] INT8 model validated. All top-1 predictions match FP32.


In [7]:
# Cell 7 - Quick accuracy check on val split
# Run 200 val samples through both FP32 and INT8
# Compare accuracy to confirm INT8 drop is acceptable

VAL_SPLIT = ROOT / 'data' / 'splits' / 'val_files.txt'

with open(VAL_SPLIT) as f:
    val_lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]

random.seed(42)
sample_lines = random.sample(val_lines, min(200, len(val_lines)))

fp32_correct = 0
int8_correct = 0
total        = 0

for line in sample_lines:
    parts = line.split('|')
    if len(parts) != 3:
        continue
    wav_path  = Path(parts[0].strip())
    true_idx  = int(parts[2].strip())
    if not wav_path.exists():
        continue
    try:
        waveform, sr = torchaudio.load(str(wav_path))
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        raw    = waveform.squeeze(0).numpy()
        tensor = preprocess(raw)
        x      = tensor.squeeze(0).unsqueeze(0).numpy()

        fp32_pred = int(np.argmax(fp32_session.run(['logits'], {'input': x})[0]))
        int8_pred = int(np.argmax(int8_session.run(['logits'], {'input': x})[0]))

        if fp32_pred == true_idx: fp32_correct += 1
        if int8_pred == true_idx: int8_correct += 1
        total += 1
    except Exception:
        continue

fp32_acc  = 100.0 * fp32_correct / total
int8_acc  = 100.0 * int8_correct / total
acc_drop  = fp32_acc - int8_acc

print('=== Quick Accuracy Check (200 val samples) ===')
print(f'  FP32 accuracy : {fp32_acc:.2f}%')
print(f'  INT8 accuracy : {int8_acc:.2f}%')
print(f'  Accuracy drop : {acc_drop:.2f}%')
if acc_drop <= 2.0:
    print('  [PASS] Drop within 2% budget.')
else:
    print('  [WARN] Drop exceeds 2%. Consider increasing calibration samples.')

=== Quick Accuracy Check (200 val samples) ===
  FP32 accuracy : 99.00%
  INT8 accuracy : 99.00%
  Accuracy drop : 0.00%
  [PASS] Drop within 2% budget.


In [ ]:
# Cell 8 - Cleanup temp calibration folder
if CALIB_DIR.exists():
    shutil.rmtree(CALIB_DIR)
    print(f'Cleaned up: {CALIB_DIR}')

print()
print('=== Quantization Summary ===')
print(f'  FP32 model  : {FP32_PATH.name}  ({FP32_PATH.stat().st_size / 1e6:.2f} MB)')
print(f'  INT8 model  : {INT8_PATH.name}  ({INT8_PATH.stat().st_size / 1e6:.2f} MB)')
print(f'  Size reduction : {reduction:.1f}%')
print(f'  FP32 acc    : {fp32_acc:.2f}%')
print(f'  INT8 acc    : {int8_acc:.2f}%')
print(f'  Drop        : {acc_drop:.2f}%')
print(f'  GV parity   : {passed}/10 PASS')
print()
print('Next step: assemble deployment_package/')

Cleaned up: C:\STREAMSENSE\temp_calibration

=== Quantization Summary ===
  FP32 model  : streamsense_model_fp32.onnx  (1.19 MB)
  INT8 model  : streamsense_model_int8.onnx  (0.31 MB)
  Size reduction : 73.6%
  FP32 acc    : 99.00%
  INT8 acc    : 99.00%
  Drop        : 0.00%
  GV parity   : 10/10 PASS

Next step: assemble deployment_package/


: 